# SmartGrid Analyse détaillée d'un run consommation

Ce notebook charge automatiquement le **meilleur run** depuis le benchmark global, ou un `RUN_ID` choisi manuellement, puis produit une analyse détaillée avec :

- métriques du run
- comparaison **vs baseline naïf hebdomadaire**
- comparaison **vs ancien modèle legacy** si disponible
- vue globale sur le backtest
- zoom sur un jour d'analyse
- erreurs par heure / par jour

> Le notebook essaie d'être robuste aux différences de noms de fichiers et de colonnes entre plusieurs versions du projet.


In [ ]:
print("test")

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import math
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')
plt.style.use('seaborn-v0_8')


## 1. Configuration utilisateur

In [ ]:
# --- Réglages manuels ---
MANUAL_PROJECT_ROOT = None
RUN_ID = None                  # ex: 'consumption_mlp_20260410T034304Z'
RUN_SUMMARY_JSON = None        # si tu veux forcer un summary.json précis
BENCHMARK_CSV = None           # si tu veux forcer un CSV benchmark précis
PREFERRED_SORT_METRIC = 'RMSE' # ou 'MAE'
PLOT_LAST_N_BACKTEST_POINTS = 7 * 144  # dernière semaine à 10 minutes


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'smartgrid').exists():
            return p
    raise FileNotFoundError('Impossible de trouver la racine du projet. Renseigne MANUAL_PROJECT_ROOT.')


PROJECT_ROOT = Path(MANUAL_PROJECT_ROOT).resolve() if MANUAL_PROJECT_ROOT else find_project_root()
print('PROJECT_ROOT =', PROJECT_ROOT)


## 2. Fonctions utilitaires

In [ ]:
def first_existing(paths):
    for p in paths:
        if p and Path(p).exists():
            return Path(p)
    return None


def choose_first_col(df: pd.DataFrame, candidates: list[str], required=True) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f'Aucune colonne trouvée parmi {candidates}. Colonnes disponibles: {list(df.columns)}')
    return None


def load_json(path: Path) -> dict:
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def find_benchmark_csv(project_root: Path) -> Path | None:
    if BENCHMARK_CSV:
        p = Path(BENCHMARK_CSV)
        return p if p.exists() else None
    candidates = [
        project_root / 'artifacts' / 'benchmarks' / 'consumption_feature_variants.csv',
        project_root / 'artifacts' / 'benchmarks' / 'feature_variants.csv',
    ]
    for c in candidates:
        if c.exists():
            return c
    found = sorted((project_root / 'artifacts').glob('**/*feature*variant*.csv'))
    return found[-1] if found else None


def auto_pick_best_run(project_root: Path, sort_metric: str = 'RMSE'):
    benchmark_csv = find_benchmark_csv(project_root)
    if benchmark_csv and benchmark_csv.exists():
        bench = pd.read_csv(benchmark_csv)
        bench.columns = [str(c).strip() for c in bench.columns]
        config_col = choose_first_col(bench, ['config_name', 'config', 'experiment_name', 'name'])
        run_id_col = choose_first_col(bench, ['run_id'], required=False)
        run_dir_col = choose_first_col(bench, ['run_dir'], required=False)
        exports_dir_col = choose_first_col(bench, ['exports_dir'], required=False)
        bench = bench.rename(columns={config_col: 'config_name'})
        if run_id_col:
            bench = bench.rename(columns={run_id_col: 'run_id'})
        if run_dir_col:
            bench = bench.rename(columns={run_dir_col: 'run_dir'})
        if exports_dir_col:
            bench = bench.rename(columns={exports_dir_col: 'exports_dir'})
        if sort_metric not in bench.columns:
            sort_metric = 'MAE' if 'MAE' in bench.columns else bench.columns[0]
        best = bench.sort_values(sort_metric, ascending=True).iloc[0].to_dict()
        return benchmark_csv, best
    return None, None


def resolve_summary_path(project_root: Path) -> Path:
    if RUN_SUMMARY_JSON:
        p = Path(RUN_SUMMARY_JSON)
        if p.exists():
            return p
        raise FileNotFoundError(f'RUN_SUMMARY_JSON introuvable: {p}')

    if RUN_ID:
        candidates = [
            project_root / 'artifacts' / 'runs' / 'consumption' / RUN_ID / 'run_summary.json',
            project_root / 'artifacts' / 'runs' / RUN_ID / 'run_summary.json',
        ]
        p = first_existing(candidates)
        if p:
            return p

    benchmark_csv, best = auto_pick_best_run(project_root, PREFERRED_SORT_METRIC)
    if best:
        if best.get('run_dir') and Path(str(best['run_dir'])).exists():
            p = Path(str(best['run_dir'])) / 'run_summary.json'
            if p.exists():
                print('Run auto-sélectionné depuis benchmark:', best.get('config_name'), '|', best.get('run_id'))
                return p
        if best.get('run_id'):
            candidates = [
                project_root / 'artifacts' / 'runs' / 'consumption' / str(best['run_id']) / 'run_summary.json',
                project_root / 'artifacts' / 'runs' / str(best['run_id']) / 'run_summary.json',
            ]
            p = first_existing(candidates)
            if p:
                print('Run auto-sélectionné depuis benchmark:', best.get('config_name'), '|', best.get('run_id'))
                return p

    found = sorted((project_root / 'artifacts').glob('**/run_summary.json'))
    if found:
        print('Aucun benchmark exploitable trouvé, dernier run_summary.json utilisé.')
        return found[-1]

    legacy_fallback = project_root / 'artifacts' / 'legacy' / 'consumption' / 'imported_from_data_processed' / 'run_summary.json'
    if legacy_fallback.exists():
        print('Fallback legacy utilisé:', legacy_fallback)
        return legacy_fallback

    raise FileNotFoundError('Impossible de trouver un run_summary.json exploitable.')


def find_history_csv(project_root: Path) -> Path | None:
    candidates = [
        project_root / 'data' / 'processed' / 'conso' / '2025-12-02_10_15_previsions_data_conso_historiques_clean.csv',
        project_root / 'data' / 'processed' / 'conso' / 'Historique 2020-2025.csv',
    ]
    p = first_existing(candidates)
    if p:
        return p
    found = sorted((project_root / 'data').glob('**/*historiques*clean*.csv'))
    return found[-1] if found else None


def load_history_total(history_csv: Path) -> pd.DataFrame:
    df = pd.read_csv(history_csv)
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)
    total_candidates = [
        'Ptot_TOTAL', 'tot', 'Ptot_TOTAL_Real'
    ]
    if any(c in df.columns for c in total_candidates):
        total_col = choose_first_col(df, total_candidates)
        out = df[['Date', total_col]].copy().rename(columns={total_col: 'real_total'})
        return out
    building_cols = ['Ptot_HA', 'Ptot_HEI_13RT', 'Ptot_HEI_5RNS', 'Ptot_RIZOMM']
    if all(c in df.columns for c in building_cols):
        out = df[['Date'] + building_cols].copy()
        out['real_total'] = out[building_cols].sum(axis=1)
        return out[['Date', 'real_total']]
    raise KeyError(f'Impossible de reconstruire le total réel depuis {history_csv}. Colonnes: {list(df.columns)}')


def compute_weekly_naive_for_dates(target_dates: pd.Series, history_total: pd.DataFrame) -> pd.Series:
    hist = history_total.copy()
    hist['Date'] = pd.to_datetime(hist['Date'])
    hist_map = hist.set_index('Date')['real_total']
    prev_dates = pd.to_datetime(target_dates) - pd.Timedelta(days=7)
    return prev_dates.map(hist_map)


def compute_metrics(real: pd.Series, pred: pd.Series) -> dict:
    mask = real.notna() & pred.notna()
    y = real[mask].astype(float)
    yhat = pred[mask].astype(float)
    if len(y) == 0:
        return {'count': 0}
    err = yhat - y
    abs_err = err.abs()
    ape = abs_err / y.replace(0, np.nan).abs()
    smape = abs_err / ((y.abs() + yhat.abs()) / 2).replace(0, np.nan)
    tol = np.maximum(0.01 * y.abs(), 5000.0)
    ramp = (yhat.diff() - y.diff()).dropna()

    out = {
        'count': int(mask.sum()),
        'MAE': float(abs_err.mean()),
        'RMSE': float(np.sqrt((err ** 2).mean())),
        'Bias(ME)': float(err.mean()),
        'MAPE%': float(100 * ape.dropna().mean()) if ape.dropna().size else np.nan,
        'SMAPE%': float(100 * smape.dropna().mean()) if smape.dropna().size else np.nan,
        'InTolerance%': float(100 * (abs_err <= tol).mean()),
        'P95AbsError': float(abs_err.quantile(0.95)),
        'P99AbsError': float(abs_err.quantile(0.99)),
        'UnderShare%': float(100 * (err < 0).mean()),
        'OverShare%': float(100 * (err > 0).mean()),
        'Under_MAE': float(abs_err[err < 0].mean()) if (err < 0).any() else np.nan,
        'Over_MAE': float(abs_err[err > 0].mean()) if (err > 0).any() else np.nan,
        'CorrAbsErr_vs_Real': float(abs_err.corr(y)) if len(y) > 1 else np.nan,
        'RampingError_RMSE': float(np.sqrt((ramp ** 2).mean())) if len(ramp) else np.nan,
    }
    return out


def infer_cols(df: pd.DataFrame):
    date_col = choose_first_col(df, ['Date', 'date', 'timestamp'])
    real_col = choose_first_col(df, ['Ptot_TOTAL_Real', 'Ptot_TOTAL', 'tot', 'actual', 'y_true'])
    pred_col = choose_first_col(df, ['Ptot_TOTAL_Forecast', 'forecast', 'prediction', 'y_pred'])
    legacy_col = choose_first_col(df, ['OldLegacy_TOTAL_Forecast', 'old_legacy_forecast', 'legacy_forecast'], required=False)
    baseline_col = choose_first_col(df, ['NaiveWeekly_Forecast', 'Baseline_Forecast', 'weekly_naive_forecast'], required=False)
    return date_col, real_col, pred_col, legacy_col, baseline_col


## 3. Chargement du run choisi

In [ ]:
SUMMARY_PATH = resolve_summary_path(PROJECT_ROOT)
summary = load_json(SUMMARY_PATH)
summary.keys()


In [ ]:
RUN_DIR = SUMMARY_PATH.parent

EXPORTS_DIR = first_existing([
    summary.get('exports_dir'),
    PROJECT_ROOT / 'artifacts' / 'exports' / 'consumption' / summary.get('run_id', ''),
])

backtest_csv = first_existing([
    summary.get('backtest_csv'),
    RUN_DIR / 'backtest.csv',
    RUN_DIR / 'backtest_predictions.csv',
    EXPORTS_DIR / 'backtest.csv' if EXPORTS_DIR else None,
    PROJECT_ROOT / 'artifacts' / 'legacy' / 'consumption' / 'imported_from_data_processed' / 'backtest_predictions.csv',
])

total_csv = first_existing([
    summary.get('output_total_csv'),
    RUN_DIR / 'notebook_total_export.csv',
    RUN_DIR / 'total_export.csv',
    EXPORTS_DIR / 'total_forecast_consumption.csv' if EXPORTS_DIR else None,
    PROJECT_ROOT / 'artifacts' / 'legacy' / 'consumption' / 'imported_from_data_processed' / 'total_legacy_v4_day.csv',
])

day_csv = first_existing([
    summary.get('day_compare_csv'),
    EXPORTS_DIR / f"selected_day_{summary.get('selected_analysis_day', '')}.csv" if EXPORTS_DIR else None,
    PROJECT_ROOT / 'artifacts' / 'legacy' / 'consumption' / 'imported_from_data_processed' / f"selected_day_{summary.get('selected_analysis_day', '')}.csv",
])

history_csv = find_history_csv(PROJECT_ROOT)

print('SUMMARY_PATH =', SUMMARY_PATH)
print('RUN_DIR =', RUN_DIR)
print('EXPORTS_DIR =', EXPORTS_DIR)
print('backtest_csv =', backtest_csv)
print('total_csv =', total_csv)
print('day_csv =', day_csv)
print('history_csv =', history_csv)


## 4. Résumé brut du run

In [ ]:
summary_df = pd.DataFrame({k: [v] for k, v in summary.items() if not isinstance(v, (dict, list))})
summary_df.T


In [ ]:
def flatten_metrics(summary: dict) -> pd.DataFrame:
    rows = []
    for key in ['metrics_model', 'metrics_naive_weekly', 'metrics_old_legacy', 'metrics_model_on_old_overlap']:
        if key in summary and isinstance(summary[key], dict):
            row = {'section': key}
            row.update(summary[key])
            rows.append(row)
    return pd.DataFrame(rows)

metrics_table = flatten_metrics(summary)
metrics_table


## 5. Chargement du backtest et ajout du baseline naïf hebdomadaire par timestamp

In [ ]:
if backtest_csv is None:
    raise FileNotFoundError('Aucun backtest CSV trouvé pour ce run.')

backtest = pd.read_csv(backtest_csv)
date_col, real_col, pred_col, legacy_col, baseline_col = infer_cols(backtest)
backtest[date_col] = pd.to_datetime(backtest[date_col], errors='coerce')
backtest = backtest.dropna(subset=[date_col]).sort_values(date_col).reset_index(drop=True)

if baseline_col is None:
    if history_csv is None:
        print('Pas de history_csv trouvé : baseline naïf hebdomadaire non reconstruit.')
    else:
        hist_total = load_history_total(history_csv)
        backtest['NaiveWeekly_Forecast'] = compute_weekly_naive_for_dates(backtest[date_col], hist_total)
        baseline_col = 'NaiveWeekly_Forecast'

backtest = backtest.rename(columns={date_col: 'Date', real_col: 'REAL', pred_col: 'MODEL'})
if legacy_col:
    backtest = backtest.rename(columns={legacy_col: 'OLD_LEGACY'})
if baseline_col:
    backtest = backtest.rename(columns={baseline_col: 'BASELINE_WEEKLY'})

backtest.head()


## 6. Comparaison globale des métriques sur le backtest

In [ ]:
comparison_rows = []
for label, col in [('MODEL', 'MODEL'), ('BASELINE_WEEKLY', 'BASELINE_WEEKLY'), ('OLD_LEGACY', 'OLD_LEGACY')]:
    if col in backtest.columns:
        row = {'model': label}
        row.update(compute_metrics(backtest['REAL'], backtest[col]))
        comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


In [ ]:
# Gains du modèle principal vs baseline et legacy
model_row = comparison_df.loc[comparison_df['model'] == 'MODEL'].iloc[0] if (comparison_df['model'] == 'MODEL').any() else None
extra = []
for ref_name in ['BASELINE_WEEKLY', 'OLD_LEGACY']:
    if (comparison_df['model'] == ref_name).any() and model_row is not None:
        ref = comparison_df.loc[comparison_df['model'] == ref_name].iloc[0]
        extra.append({
            'vs': ref_name,
            'MAE_skill_%': 100 * (1 - model_row['MAE'] / ref['MAE']) if ref['MAE'] else np.nan,
            'RMSE_skill_%': 100 * (1 - model_row['RMSE'] / ref['RMSE']) if ref['RMSE'] else np.nan,
            'MAPE_delta': model_row['MAPE%'] - ref['MAPE%'] if 'MAPE%' in comparison_df.columns else np.nan,
            'SMAPE_delta': model_row['SMAPE%'] - ref['SMAPE%'] if 'SMAPE%' in comparison_df.columns else np.nan,
            'InTolerance_delta': model_row['InTolerance%'] - ref['InTolerance%'] if 'InTolerance%' in comparison_df.columns else np.nan,
            'Ramping_delta': model_row['RampingError_RMSE'] - ref['RampingError_RMSE'] if 'RampingError_RMSE' in comparison_df.columns else np.nan,
        })

gains_df = pd.DataFrame(extra)
gains_df


## 7. Vue globale du backtest

In [ ]:
plot_df = backtest.tail(PLOT_LAST_N_BACKTEST_POINTS).copy()
plt.figure(figsize=(18, 6))
plt.plot(plot_df['Date'], plot_df['REAL'], label='Réel')
plt.plot(plot_df['Date'], plot_df['MODEL'], label='Nouveau modèle')
if 'BASELINE_WEEKLY' in plot_df.columns:
    plt.plot(plot_df['Date'], plot_df['BASELINE_WEEKLY'], label='Baseline weekly', alpha=0.9)
if 'OLD_LEGACY' in plot_df.columns:
    plt.plot(plot_df['Date'], plot_df['OLD_LEGACY'], label='Ancien legacy', alpha=0.9)
plt.title('Backtest — dernière fenêtre')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Distribution des erreurs absolues
err_df = pd.DataFrame({
    'MODEL': (backtest['MODEL'] - backtest['REAL']).abs(),
})
if 'BASELINE_WEEKLY' in backtest.columns:
    err_df['BASELINE_WEEKLY'] = (backtest['BASELINE_WEEKLY'] - backtest['REAL']).abs()
if 'OLD_LEGACY' in backtest.columns:
    err_df['OLD_LEGACY'] = (backtest['OLD_LEGACY'] - backtest['REAL']).abs()

err_df.plot(kind='box', figsize=(12, 5), title='Distribution des erreurs absolues')
plt.tight_layout()
plt.show()


## 8. Analyse sur le jour sélectionné

In [ ]:
selected_day = summary.get('selected_analysis_day')
if day_csv and Path(day_csv).exists():
    day_df = pd.read_csv(day_csv)
    dcol, rcol, pcol, lcol, bcol = infer_cols(day_df)
    day_df[dcol] = pd.to_datetime(day_df[dcol], errors='coerce')
    day_df = day_df.dropna(subset=[dcol]).sort_values(dcol).reset_index(drop=True)
    day_df = day_df.rename(columns={dcol: 'Date', rcol: 'REAL', pcol: 'MODEL'})
    if lcol:
        day_df = day_df.rename(columns={lcol: 'OLD_LEGACY'})
    if bcol:
        day_df = day_df.rename(columns={bcol: 'BASELINE_WEEKLY'})
    elif history_csv:
        hist_total = load_history_total(history_csv)
        day_df['BASELINE_WEEKLY'] = compute_weekly_naive_for_dates(day_df['Date'], hist_total)
else:
    if not selected_day:
        selected_day = backtest['Date'].dt.date.max().isoformat()
    day_mask = backtest['Date'].dt.date == pd.to_datetime(selected_day).date()
    day_df = backtest.loc[day_mask].copy()

print('selected_day =', selected_day)
day_df.head()


In [ ]:
plt.figure(figsize=(18, 6))
plt.plot(day_df['Date'], day_df['REAL'], label='Réel', linewidth=2)
plt.plot(day_df['Date'], day_df['MODEL'], label='Nouveau modèle', linewidth=2)
if 'BASELINE_WEEKLY' in day_df.columns:
    plt.plot(day_df['Date'], day_df['BASELINE_WEEKLY'], label='Baseline weekly')
if 'OLD_LEGACY' in day_df.columns:
    plt.plot(day_df['Date'], day_df['OLD_LEGACY'], label='Ancien legacy')
plt.title(f'Jour d'analyse — {selected_day}')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
day_rows = []
for label, col in [('MODEL', 'MODEL'), ('BASELINE_WEEKLY', 'BASELINE_WEEKLY'), ('OLD_LEGACY', 'OLD_LEGACY')]:
    if col in day_df.columns:
        row = {'model': label}
        row.update(compute_metrics(day_df['REAL'], day_df[col]))
        day_rows.append(row)

day_metrics_df = pd.DataFrame(day_rows)
day_metrics_df


## 9. Erreurs par heure et par jour

In [ ]:
work = backtest.copy()
work['hour'] = work['Date'].dt.hour
work['date_only'] = work['Date'].dt.date
work['MODEL_abs_err'] = (work['MODEL'] - work['REAL']).abs()
if 'BASELINE_WEEKLY' in work.columns:
    work['BASELINE_abs_err'] = (work['BASELINE_WEEKLY'] - work['REAL']).abs()
if 'OLD_LEGACY' in work.columns:
    work['OLD_abs_err'] = (work['OLD_LEGACY'] - work['REAL']).abs()

hourly = work.groupby('hour')[['MODEL_abs_err'] + [c for c in ['BASELINE_abs_err', 'OLD_abs_err'] if c in work.columns]].mean()
daily = work.groupby('date_only')[['MODEL_abs_err'] + [c for c in ['BASELINE_abs_err', 'OLD_abs_err'] if c in work.columns]].mean()

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
hourly.plot(ax=axes[0], marker='o', title='Erreur absolue moyenne par heure')
daily.tail(30).plot(ax=axes[1], title='Erreur absolue moyenne par jour — 30 derniers jours')
plt.tight_layout()
plt.show()


## 10. À raconter dans la présentation

- Le **benchmark global** sert à choisir les meilleurs candidats.
- Ce notebook montre ensuite le **comportement réel** du meilleur run :
  - précision globale,
  - comparaison au baseline naïf hebdomadaire,
  - comparaison à l'ancien modèle si disponible,
  - comportement intraday,
  - gestion des pics et des variations rapides.

Tu peux changer `RUN_ID` en haut si tu veux analyser un autre run sélectionné par le benchmark.
